<a href="https://colab.research.google.com/github/barr92/public-house-price-data/blob/main/signalsbi_rental_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Signals BI — Rental Listings Pipeline
**Project:** `signalsbi-new` | **Dataset:** `signals_rents`

## How to use
| Scenario | Settings |
|---|---|
| First ever run | `ENV = 'dev'`, `FIRST_RUN = True` |
| Monthly update (append new files) | `ENV = 'prod'`, `FIRST_RUN = False` |
| Promote dev → prod | Run Cell 8 with `PROMOTE = True` |

## Tables produced
| Table | Description |
|---|---|
| `raw_rental_listings` | Append-only, every listing from every scrape, unified schema |
| `rental_status_log` | One row per status change per listing (active → let agreed) |
| `cleaned_rental_listings` | Latest status per listing, deduplicated, price cleaned |
| `rental_sector_latest` | Latest aggregation per sector × bedroom band |
| `rental_sector_by_type` | Aggregation per sector × bedroom band × property type |
| `rental_trends_monthly` | Monthly trends per sector × bedroom band |

## 0. Apify webhook trigger (future use)
Commented out — will trigger Rightmove scraper automatically once configured.

In [1]:
# ── FUTURE: Apify webhook trigger ──────────────────────────────────────────
# Uncomment when Apify webhook is configured.
#
# import requests
#
# APIFY_TOKEN = 'your_apify_token_here'
# RIGHTMOVE_ACTOR_ID = 'your_rightmove_actor_id'
#
# def trigger_scraper(actor_id):
#     url = f'https://api.apify.com/v2/acts/{actor_id}/runs?token={APIFY_TOKEN}'
#     resp = requests.post(url, json={'memory': 4096})
#     resp.raise_for_status()
#     run_id = resp.json()['data']['id']
#     print(f'Started run {run_id}')
#     return run_id
#
# trigger_scraper(RIGHTMOVE_ACTOR_ID)
# ─────────────────────────────────────────────────────────────────────────────

print('Apify trigger skipped (manual mode)')

Apify trigger skipped (manual mode)


## 1. Authenticate

In [2]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


## 2. Install & import

In [ ]:
# !pip install google-cloud-bigquery google-cloud-bigquery-storage google-cloud-storage pandas db-dtypes --quiet

import io
import re
import pandas as pd
from datetime import datetime, timezone
from google.cloud import bigquery, storage

print('Imports OK')

## 3. Configuration
**Only cell you need to change between runs.**

In [ ]:
# ── CHANGE THESE BETWEEN RUNS ────────────────────────────────────────────────
ENV       = 'dev'   # 'dev' or 'prod'
FIRST_RUN = True    # True = ingest all GCS files | False = only new files
# ─────────────────────────────────────────────────────────────────────────────

PROJECT   = 'signalsbi-new'
DATASETS  = {'prod': 'signals_rents', 'dev': 'signals_rents_dev'}
DATASET   = DATASETS[ENV]
D         = f'`{PROJECT}.{DATASET}`'
GCS_BUCKET = 'signals-new'
GCS_PREFIX = 'asking prices/Source/'

# Skip these file patterns
SKIP_PATTERNS = [
    'asking_prices_Source_dataset_rental-listings-uk-2025-04-24',  # already in prod
]

bq  = bigquery.Client(project=PROJECT, location='EU')
gcs = storage.Client(project=PROJECT)

print(f'Environment : {ENV.upper()}')
print(f'First run   : {FIRST_RUN}')
print(f'Dataset     : {PROJECT}.{DATASET}')

## 4. Create dataset and tables

In [ ]:
ds = bigquery.Dataset(f'{PROJECT}.{DATASET}')
ds.location = 'EU'
bq.create_dataset(ds, exists_ok=True)
print(f'Dataset `{PROJECT}.{DATASET}` ready.')

ddls = [

# 1. ingested_files — tracks which GCS files have been processed
f'''
CREATE TABLE IF NOT EXISTS {D}.ingested_files (
  file_name     STRING NOT NULL,
  gcs_path      STRING,
  source        STRING,
  row_count     INT64,
  ingested_at   TIMESTAMP
)
''',

# 2. raw_rental_listings — append-only, unified schema
f'''
CREATE OR REPLACE TABLE {D}.raw_rental_listings (
  listing_id        STRING,
  source            STRING,
  address_full      STRING,
  postcode          STRING,
  postcode_sector   STRING,
  postcode_district STRING,
  town              STRING,
  latitude          FLOAT64,
  longitude         FLOAT64,
  price_pcm         INT64,
  price_pw          INT64,
  bedrooms          INT64,
  bathrooms         INT64,
  property_type     STRING,
  bedroom_band      STRING,
  furnished         STRING,
  is_let_agreed     BOOL,
  is_new            BOOL,
  is_reduced        BOOL,
  listing_status    STRING,
  listing_date      DATE,
  available_from    STRING,
  agent_name        STRING,
  source_url        STRING,
  scraped_at        TIMESTAMP,
  ingested_at       TIMESTAMP,
  file_name         STRING
)
PARTITION BY listing_date
CLUSTER BY postcode_sector, property_type, bedroom_band
OPTIONS (require_partition_filter = false)
''',

# 3. rental_status_log — status change tracking
f'''
CREATE OR REPLACE TABLE {D}.rental_status_log (
  listing_id        STRING,
  previous_status   STRING,
  new_status        STRING,
  changed_at        TIMESTAMP,
  file_name         STRING,
  source            STRING
)
'''
]

for ddl in ddls:
    bq.query(ddl).result()

print('Tables ready.')

## 5. Discover and ingest GCS files
Handles two schemas:
- **Old format** (home.co.uk / asking prices) — camelCase columns matching existing `raw_rental_listings`
- **New format** (Rightmove scraper) — 2000 nested columns, different field names

In [ ]:
def derive_bedroom_band(bedrooms):
    if bedrooms is None: return 'Unknown'
    try:
        b = int(bedrooms)
        if b <= 0: return 'Studio'
        if b == 1: return '1 bed'
        if b == 2: return '2 bed'
        if b == 3: return '3 bed'
        if b == 4: return '4 bed'
        return '5+ bed'
    except: return 'Unknown'

def derive_postcode_sector(postcode):
    pc = str(postcode or '').strip().upper()
    if ' ' in pc:
        parts = pc.split(' ')
        if len(parts[1]) >= 1:
            return parts[0] + ' ' + parts[1][0]
    return pc

def safe_int(val):
    try: return int(float(val)) if val and str(val).lower() not in ['nan','none',''] else None
    except: return None

def safe_float(val):
    try: return float(val) if val and str(val).lower() not in ['nan','none',''] else None
    except: return None

def safe_bool(val):
    if isinstance(val, bool): return val
    if str(val).lower() in ['true','1','yes']: return True
    if str(val).lower() in ['false','0','no']: return False
    return None

def safe_date(val):
    if not val or str(val).lower() in ['nan','none','']: return None
    try: return pd.to_datetime(str(val)[:10]).date()
    except: return None

def is_rightmove_format(df):
    """Detect new Rightmove scraper format by checking for its unique columns."""
    return 'addedOn' in df.columns and 'primaryPrice' in df.columns

def extract_postcode_from_address(address):
    """Extract postcode from address string using regex."""
    if not address: return None
    match = re.search(r'([A-Z]{1,2}\d{1,2}[A-Z]?\s*\d[A-Z]{2})', str(address).upper())
    return match.group(1).strip() if match else None

def normalise_old_format(row, file_name, ingested_at):
    """Normalise old home.co.uk / asking prices format."""
    r = dict(row)
    g = lambda k: r.get(k) or r.get(k.lower())

    postcode = str(g('postcode') or '').strip().upper()
    bedrooms = safe_int(g('bedrooms'))

    return {
        'listing_id':        str(g('listingId') or ''),
        'source':            'home.co.uk',
        'address_full':      str(g('addressFull') or g('_addressFull_') or ''),
        'postcode':          postcode,
        'postcode_sector':   derive_postcode_sector(postcode),
        'postcode_district': postcode.split(' ')[0] if ' ' in postcode else postcode[:4].strip(),
        'town':              str(g('location') or ''),
        'latitude':          safe_float(g('latitude')),
        'longitude':         safe_float(g('longitude')),
        'price_pcm':         safe_int(g('pricePcm')),
        'price_pw':          safe_int(g('pricePw')),
        'bedrooms':          bedrooms,
        'bathrooms':         safe_int(g('bathrooms')),
        'property_type':     str(g('propertyType') or ''),
        'bedroom_band':      derive_bedroom_band(bedrooms),
        'furnished':         str(g('furnished') or ''),
        'is_let_agreed':     safe_bool(g('isLetAgreed')),
        'is_new':            safe_bool(g('isNew')),
        'is_reduced':        safe_bool(g('isReduced')),
        'listing_status':    str(g('listingStatus') or ''),
        'listing_date':      safe_date(g('listingDate')),
        'available_from':    str(g('availableFrom') or ''),
        'agent_name':        str(g('agentName') or ''),
        'source_url':        str(g('sourceUrl') or ''),
        'scraped_at':        pd.to_datetime(g('scrapedAt'), errors='coerce', utc=True),
        'ingested_at':       ingested_at,
        'file_name':         file_name,
    }

def normalise_rightmove_format(row, file_name, ingested_at):
    """Normalise new Rightmove scraper format (2000 nested columns)."""
    r = dict(row)
    g = lambda k: r.get(k)

    # Extract price — primaryPrice is like '£1,725 pcm'
    price_raw = str(g('primaryPrice') or '')
    price_match = re.search(r'[\d,]+', price_raw.replace(',',''))
    price_pcm = safe_int(re.sub(r'[^\d]', '', price_raw)) if price_raw else None

    # Address and postcode
    address = str(g('address') or '')
    postcode = extract_postcode_from_address(address) or ''
    postcode = postcode.strip().upper()

    # Let agreed status — buried in analytics
    let_agreed_raw = g('lettings/minimumTermInMonths/analyticsInfo/analyticsProperty/letAgreed')
    is_let_agreed = safe_bool(let_agreed_raw)

    # Listing date from addedOn e.g. '28/05/2026'
    added_on = str(g('addedOn') or '')
    listing_date = None
    if added_on:
        try: listing_date = pd.to_datetime(added_on, dayfirst=True).date()
        except: pass

    bedrooms = safe_int(g('beds'))
    prop_type = str(g('propertyType') or g('propertySubType') or '')

    return {
        'listing_id':        str(g('id') or ''),
        'source':            'rightmove',
        'address_full':      address,
        'postcode':          postcode,
        'postcode_sector':   derive_postcode_sector(postcode),
        'postcode_district': postcode.split(' ')[0] if ' ' in postcode else postcode[:4].strip(),
        'town':              str(g('lettings/minimumTermInMonths/analyticsInfo/analyticsBranch/displayAddress') or ''),
        'latitude':          safe_float(g('latitude')),
        'longitude':         safe_float(g('longitude')),
        'price_pcm':         price_pcm,
        'price_pw':          None,
        'bedrooms':          bedrooms,
        'bathrooms':         safe_int(g('baths')),
        'property_type':     prop_type,
        'bedroom_band':      derive_bedroom_band(bedrooms),
        'furnished':         str(g('lettings/furnishType') or g('furnishedType') or ''),
        'is_let_agreed':     is_let_agreed,
        'is_new':            safe_bool(g('lettings/minimumTermInMonths/analyticsInfo/analyticsProperty/preOwned')),
        'is_reduced':        None,
        'listing_status':    'let_agreed' if is_let_agreed else 'active',
        'listing_date':      listing_date,
        'available_from':    str(g('lettings/letAvailableDate') or ''),
        'agent_name':        str(g('branch/companyName') or g('branch/brandName') or ''),
        'source_url':        str(g('url') or ''),
        'scraped_at':        pd.Timestamp.now(tz='UTC'),
        'ingested_at':       ingested_at,
        'file_name':         file_name,
    }

def process_csv(content_bytes, file_name, ingested_at):
    df_raw = pd.read_csv(io.BytesIO(content_bytes), dtype=str, low_memory=False)
    print(f'  {len(df_raw):,} raw rows')

    if is_rightmove_format(df_raw):
        print('  Format: Rightmove (new)')
        rows = [normalise_rightmove_format(row, file_name, ingested_at)
                for _, row in df_raw.iterrows()]
    else:
        print('  Format: home.co.uk (old)')
        rows = [normalise_old_format(row, file_name, ingested_at)
                for _, row in df_raw.iterrows()]

    df = pd.DataFrame(rows)
    df = df[df['listing_id'].str.len() > 0]
    df = df[df['price_pcm'].notna()]
    df = df[df['price_pcm'] > 0]
    df = df.drop_duplicates(subset=['listing_id'], keep='last')
    print(f'  {len(df):,} clean rows after dedup')
    return df

def get_ingested_files():
    try:
        return set(bq.query(f'SELECT file_name FROM {D}.ingested_files')
                     .to_dataframe()['file_name'].tolist())
    except: return set()

def get_existing_listing_ids():
    try:
        return set(bq.query(f'SELECT DISTINCT listing_id FROM {D}.raw_rental_listings')
                     .to_dataframe()['listing_id'].tolist())
    except: return set()

def log_ingested_file(file_name, gcs_path, source, row_count, ingested_at):
    df = pd.DataFrame([{'file_name': file_name, 'gcs_path': gcs_path,
                         'source': source, 'row_count': row_count,
                         'ingested_at': ingested_at}])
    bq.load_table_from_dataframe(df, f'{PROJECT}.{DATASET}.ingested_files',
        job_config=bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')).result()

# ── Main ingest loop ──────────────────────────────────────────────────────────
bucket = gcs.bucket(GCS_BUCKET)
blobs  = list(bucket.list_blobs(prefix=GCS_PREFIX))
already_ingested  = set() if FIRST_RUN else get_ingested_files()
existing_ids      = get_existing_listing_ids()

print(f'Found {len(blobs)} files in GCS')
print(f'Already ingested: {len(already_ingested)} files')
print(f'Existing listing IDs in BQ: {len(existing_ids):,}')

total_new = 0
for blob in blobs:
    file_name = blob.name.split('/')[-1]
    if not file_name.endswith('.csv'):
        continue
    if file_name in already_ingested:
        print(f'  SKIP (already ingested): {file_name}')
        continue
    if any(p in file_name for p in SKIP_PATTERNS):
        print(f'  SKIP (in skip list): {file_name}')
        continue

    print(f'\nProcessing: {file_name}')
    content = blob.download_as_bytes()
    ingested_at = datetime.now(timezone.utc)
    df = process_csv(content, file_name, ingested_at)

    # Only append listings not already in BQ
    df_new = df[~df['listing_id'].isin(existing_ids)]
    print(f'  {len(df_new):,} new listings (not in BQ already)')

    if len(df_new) > 0:
        bq.load_table_from_dataframe(
            df_new, f'{PROJECT}.{DATASET}.raw_rental_listings',
            job_config=bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
        ).result()
        existing_ids.update(df_new['listing_id'].tolist())
        total_new += len(df_new)

    log_ingested_file(file_name, blob.name, df['source'].iloc[0] if len(df) else 'unknown',
                      len(df_new), ingested_at)
    print(f'  Done.')

n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.raw_rental_listings').to_dataframe().iloc[0]['n']
print(f'\nTotal new rows added: {total_new:,}')
print(f'raw_rental_listings total: {n:,}')

## 6. Build rental status log
Detects when a listing moves from active → let agreed.

In [ ]:
print('Building rental_status_log...')
bq.query(f'''
INSERT INTO {D}.rental_status_log
  (listing_id, previous_status, new_status, changed_at, file_name, source)

WITH ordered AS (
  SELECT
    listing_id, listing_status, is_let_agreed, ingested_at, file_name, source,
    LAG(listing_status) OVER (PARTITION BY listing_id ORDER BY ingested_at) AS prev_status,
    LAG(is_let_agreed)  OVER (PARTITION BY listing_id ORDER BY ingested_at) AS prev_let_agreed
  FROM {D}.raw_rental_listings
  WHERE listing_id IS NOT NULL AND listing_id != ''
),
changes AS (
  SELECT * FROM ordered
  WHERE prev_status IS NOT NULL
    AND (listing_status != prev_status OR is_let_agreed != prev_let_agreed)
),
already_logged AS (
  SELECT listing_id, new_status, changed_at FROM {D}.rental_status_log
)
SELECT
  c.listing_id,
  c.prev_status    AS previous_status,
  c.listing_status AS new_status,
  c.ingested_at    AS changed_at,
  c.file_name,
  c.source
FROM changes c
LEFT JOIN already_logged al
  ON c.listing_id = al.listing_id
  AND c.listing_status = al.new_status
  AND c.ingested_at = al.changed_at
WHERE al.listing_id IS NULL
''').result()

n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.rental_status_log').to_dataframe().iloc[0]['n']
print(f'rental_status_log: {n:,} change events')

## 7. Build `cleaned_rental_listings`
One row per listing, latest status, P2–P98 price cleaning by postcode district.

In [ ]:
print('Building cleaned_rental_listings...')
bq.query(f'''
CREATE OR REPLACE TABLE {D}.cleaned_rental_listings AS

WITH latest AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY listing_id
      ORDER BY ingested_at DESC
    ) AS rn
  FROM {D}.raw_rental_listings
  WHERE listing_id IS NOT NULL AND listing_id != ''
    AND price_pcm IS NOT NULL AND price_pcm > 0
),
deduped AS (
  SELECT * FROM latest WHERE rn = 1
),
-- P2/P98 price cleaning by postcode district + bedroom band
price_bounds AS (
  SELECT
    postcode_district,
    bedroom_band,
    APPROX_QUANTILES(price_pcm, 100)[OFFSET(2)]  AS p2,
    APPROX_QUANTILES(price_pcm, 100)[OFFSET(98)] AS p98
  FROM deduped
  GROUP BY 1, 2
)
SELECT
  d.listing_id,
  d.source,
  d.postcode,
  d.postcode_sector,
  d.postcode_district,
  d.listing_date,
  d.price_pcm,
  d.bedroom_band,
  d.property_type,
  d.latitude,
  d.longitude,
  d.is_let_agreed,
  d.listing_status,
  d.furnished,
  d.bedrooms,
  d.bathrooms,
  d.agent_name,
  d.source_url,
  d.ingested_at
FROM deduped d
JOIN price_bounds p
  ON d.postcode_district = p.postcode_district
  AND d.bedroom_band = p.bedroom_band
WHERE d.price_pcm BETWEEN p.p2 AND p.p98
''').result()

n = bq.query(f'SELECT COUNT(*) AS n FROM {D}.cleaned_rental_listings').to_dataframe().iloc[0]['n']
print(f'cleaned_rental_listings: {n:,} unique listings')

## 8. Build aggregated tables

In [ ]:
def run_query(name, sql):
    print(f'  Refreshing {name}...', end=' ', flush=True)
    bq.query(sql).result()
    print('done')

run_query('rental_sector_latest', f'''
CREATE OR REPLACE TABLE {D}.rental_sector_latest AS

WITH latest_month AS (
  SELECT MAX(FORMAT_DATE('%Y-%m', listing_date)) AS ym
  FROM {D}.cleaned_rental_listings
  WHERE listing_date IS NOT NULL
),
recent AS (
  SELECT *
  FROM {D}.cleaned_rental_listings
  WHERE FORMAT_DATE('%Y-%m', listing_date) = (SELECT ym FROM latest_month)
    AND is_let_agreed = FALSE
)
SELECT
  postcode_sector,
  bedroom_band,
  'latest_month'                                  AS data_window,
  PARSE_DATE('%Y-%m', (SELECT ym FROM latest_month)) AS reference_month,
  COUNT(*)                                        AS listing_count,
  CAST(APPROX_QUANTILES(price_pcm, 100)[OFFSET(50)] AS INT64) AS median_pcm,
  CAST(ROUND(AVG(price_pcm), 0) AS INT64)         AS mean_pcm,
  CAST(MIN(price_pcm) AS INT64)                   AS min_pcm,
  CAST(MAX(price_pcm) AS INT64)                   AS max_pcm,
  CAST(APPROX_QUANTILES(price_pcm, 100)[OFFSET(25)] AS INT64) AS p25_pcm,
  CAST(APPROX_QUANTILES(price_pcm, 100)[OFFSET(75)] AS INT64) AS p75_pcm,
  CURRENT_TIMESTAMP()                             AS processed_at
FROM recent
WHERE postcode_sector IS NOT NULL
GROUP BY 1, 2, 3, 4
''')

run_query('rental_sector_by_type', f'''
CREATE OR REPLACE TABLE {D}.rental_sector_by_type AS

WITH latest_month AS (
  SELECT MAX(FORMAT_DATE('%Y-%m', listing_date)) AS ym
  FROM {D}.cleaned_rental_listings
  WHERE listing_date IS NOT NULL
),
recent AS (
  SELECT *
  FROM {D}.cleaned_rental_listings
  WHERE FORMAT_DATE('%Y-%m', listing_date) = (SELECT ym FROM latest_month)
    AND is_let_agreed = FALSE
)
SELECT
  postcode_sector,
  bedroom_band,
  property_type,
  COUNT(*)                                        AS listing_count,
  CAST(APPROX_QUANTILES(price_pcm, 100)[OFFSET(50)] AS INT64) AS median_pcm,
  CAST(ROUND(AVG(price_pcm), 0) AS INT64)         AS mean_pcm,
  CAST(MIN(price_pcm) AS INT64)                   AS min_pcm,
  CAST(MAX(price_pcm) AS INT64)                   AS max_pcm,
  CURRENT_TIMESTAMP()                             AS processed_at
FROM recent
WHERE postcode_sector IS NOT NULL
GROUP BY 1, 2, 3
''')

run_query('rental_trends_monthly', f'''
CREATE OR REPLACE TABLE {D}.rental_trends_monthly AS

WITH monthly AS (
  SELECT
    postcode_sector,
    bedroom_band,
    FORMAT_DATE('%Y-%m', listing_date) AS year_month,
    COUNT(*)                                          AS listing_count,
    COUNTIF(is_let_agreed = TRUE)                     AS let_agreed_count,
    CAST(APPROX_QUANTILES(price_pcm, 100)[OFFSET(50)] AS INT64) AS median_pcm,
    CAST(ROUND(AVG(price_pcm), 0) AS INT64)           AS mean_pcm
  FROM {D}.cleaned_rental_listings
  WHERE listing_date IS NOT NULL
    AND postcode_sector IS NOT NULL
  GROUP BY 1, 2, 3
)
SELECT *,
  ROUND(SAFE_DIVIDE(
    mean_pcm - LAG(mean_pcm) OVER (PARTITION BY postcode_sector, bedroom_band ORDER BY year_month),
    LAG(mean_pcm) OVER (PARTITION BY postcode_sector, bedroom_band ORDER BY year_month)
  ) * 100, 1) AS mean_mom_pct,
  ROUND(SAFE_DIVIDE(
    mean_pcm - LAG(mean_pcm, 12) OVER (PARTITION BY postcode_sector, bedroom_band ORDER BY year_month),
    LAG(mean_pcm, 12) OVER (PARTITION BY postcode_sector, bedroom_band ORDER BY year_month)
  ) * 100, 1) AS mean_yoy_pct,
  CURRENT_TIMESTAMP() AS processed_at
FROM monthly
ORDER BY postcode_sector, bedroom_band, year_month
''')

## 9. Pipeline summary

In [ ]:
tables = [
    ('ingested_files',          'ingested_at'),
    ('raw_rental_listings',     'ingested_at'),
    ('rental_status_log',       'changed_at'),
    ('cleaned_rental_listings', 'ingested_at'),
    ('rental_sector_latest',    'processed_at'),
    ('rental_sector_by_type',   'processed_at'),
    ('rental_trends_monthly',   'processed_at'),
]
parts = ' UNION ALL '.join([
    f"SELECT '{t}' AS table_name, COUNT(*) AS row_count, MAX(CAST({ts} AS STRING)) AS last_updated FROM {D}.{t}"
    for t, ts in tables
])
summary = bq.query(parts + ' ORDER BY table_name').to_dataframe()
print(f'=== Rental Pipeline [{ENV.upper()}] ===')
print(f'Dataset : {PROJECT}.{DATASET}\n')
print(summary.to_string(index=False))

## 10. Promote dev → prod
Run manually after validating dev. Set `PROMOTE = True` to execute.

In [ ]:
PROMOTE = False  # <-- set True to copy dev -> prod

if not PROMOTE:
    print('Promotion skipped. Set PROMOTE = True to copy dev -> prod.')
else:
    if ENV != 'dev':
        raise RuntimeError("Set ENV = 'dev' before promoting.")
    prod_dataset = DATASETS['prod']
    to_promote = [t for t, _ in tables]
    for table in to_promote:
        src = f'{PROJECT}.{DATASETS["dev"]}.{table}'
        dst = f'{PROJECT}.{prod_dataset}.{table}'
        print(f'  Copying {table}...', end=' ', flush=True)
        try:
            bq.copy_table(src, dst, job_config=bigquery.CopyJobConfig(
                write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)).result()
            print('done')
        except Exception as e:
            print(f'SKIPPED ({e})')
    print(f'Promotion complete -> {PROJECT}.{prod_dataset}')